# Coverage decomposition under a misspecified dictionary

This notebook generates the coverage decomposition reported for the diagnostic design. The representer and outcome estimators use the same treatment-specific dictionary. The misspecified dictionary omits the centered quadratic term that appears in both the treatment index and the outcome regression. Numerical failures remain in the denominator and are reported through the stable count. Coverage, bias, and the error summaries are computed over the stable replications only; the stable and failure columns in the same table state that conditioning explicitly.


In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "src" / "genriesz").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the genriesz repository.")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"

TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
_LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")


def label_of(value):
    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def display_table(df: pd.DataFrame, *, caption: str | None = None, digits: int = 4):
    table = df.copy()
    for column in _LABEL_COLUMNS:
        if column in table.columns:
            table[column] = table[column].map(label_of)
    numeric = table.select_dtypes(include=[np.number]).columns
    table[numeric] = table[numeric].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)

pd.options.display.max_rows = TABLE_CONFIG["max_rows"]

from genriesz.experiments import CoverageDiagnosticBasis, make_coverage_diagnostic_data


In [ ]:
N_REPLICATIONS = 400
SAMPLE_SIZE = 2000
OVERLAP_SETTINGS = {"Strong": 0.5, "Weak": 2.5}
DICTIONARIES = {
    "Correct": CoverageDiagnosticBasis(include_quadratic=True),
    "Misspecified": CoverageDiagnosticBasis(include_quadratic=False),
}
RIEZ_LAMBDA = 1e-2
FOLDS = 2


In [ ]:
rows = []
for overlap_name, overlap_scale in OVERLAP_SETTINGS.items():
    for replication in range(N_REPLICATIONS):
        data = make_coverage_diagnostic_data(
            n=SAMPLE_SIZE,
            seed=1_000_000 + 10_007 * replication + int(100 * overlap_scale),
            overlap_scale=overlap_scale,
        )
        for dictionary_name, basis in DICTIONARIES.items():
            for loss_spec in COMPATIBLE_LOSSES:
                fit_rows = fit_one_grr_with_basis(
                    data,
                    estimand="ATE",
                    loss_spec=loss_spec,
                    representer_basis=basis,
                    cross_fit=True,
                    lam=RIEZ_LAMBDA,
                    folds=FOLDS,
                    estimators=("arw",),
                    random_state=replication,
                    label_info={
                        "overlap": overlap_name,
                        "dictionary": dictionary_name,
                        "replication": replication,
                    },
                )
                rows.extend(fit_rows)
coverage_results = pd.DataFrame(rows)


In [ ]:
summary_rows = []
for keys, group in coverage_results.groupby(["overlap", "dictionary", "loss"], dropna=False):
    overlap_name, dictionary_name, loss = keys
    stable = group[group["status"] == "ok"].copy()
    row = {
        "overlap": overlap_name,
        "dictionary": dictionary_name,
        "loss": loss,
        "stable": int(stable.shape[0]),
        "failures": int(group.shape[0] - stable.shape[0]),
    }
    if not stable.empty:
        row.update({
            "bias": float(stable["error"].mean()),
            "mc_sd": float(stable["estimate"].std(ddof=1)),
            "mean_se": float(stable["se"].mean()),
            "coverage": float(stable["covered"].mean()),
            "rmse": float(np.sqrt(stable["squared_error"].mean())),
            "held_out_imbalance": float(stable["held_out_imbalance_max"].mean()),
            "max_abs_alpha": float(stable["alpha_abs_max"].mean()),
            "binding_rate_max": float(stable["riesz_clip_binding_rate_max"].max()),
        })
    summary_rows.append(row)
coverage_summary = pd.DataFrame(summary_rows).sort_values(["overlap", "dictionary", "loss"])
display_table(coverage_summary, caption="Coverage decomposition under correct and misspecified dictionaries")


In [ ]:
plot_data = coverage_summary.copy()
plot_data["specification"] = plot_data["dictionary"] + " | " + plot_data["loss"]
for overlap_name in OVERLAP_SETTINGS:
    panel = plot_data[plot_data["overlap"] == overlap_name].copy()
    figure, axis = plt.subplots(figsize=(11.0, 5.2), dpi=PLOT_CONFIG["dpi"])
    positions = np.arange(panel.shape[0])
    axis.bar(positions, panel["coverage"].fillna(0.0))
    axis.axhline(0.95, color="black", linestyle="--", linewidth=1.0)
    axis.set_xticks(positions)
    axis.set_xticklabels(panel["specification"], rotation=45, ha="right")
    axis.set_ylim(0.0, 1.02)
    axis.set_ylabel("Coverage probability", fontsize=PLOT_CONFIG["axis_fontsize"])
    axis.set_title(f"{overlap_name} overlap", fontsize=PLOT_CONFIG["title_fontsize"])
    axis.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
    figure.tight_layout()
    plt.show()
